<a href="https://colab.research.google.com/github/mhowlin-web/TP_RAG_ARCA/blob/main/06_retrieval_pinecone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP RAG y Agentes

## Retrieval de documentos con Pinecone

En este notebook implemento la etapa de retrieval de mi sistema RAG.

Ya tengo el corpus de documentos de ARCA dividido en chunks y cada chunk tiene un embedding generado con `sentence-transformers/all-MiniLM-L6-v2`.

En el notebook anterior cargué esos embeddings en Pinecone.

Ahora voy a transformar una pregunta en un embedding y consultar Pinecone para recuperar los fragmentos del corpus que tienen mayor similitud semántica con esa pregunta.

Todavía no voy a utilizar un modelo generativo. Primero quiero verificar que la recuperación de información funciona correctamente.

## Instalación de librerías

En esta celda instalo las librerías que necesito para conectarme con Pinecone y generar embeddings para las preguntas.

In [2]:
!pip install -q pinecone sentence-transformers

## Importación de librerías

En esta celda importo las librerías que voy a utilizar durante el proceso de retrieval.

También utilizo los Secrets de Google Colab para obtener de manera segura la API key de Pinecone.

In [4]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from google.colab import userdata

## Carga de la API key de Pinecone

En esta celda recupero la API key desde los Secrets de Google Colab.

No escribo la API key directamente en el notebook porque este proyecto será versionado en GitHub.

In [5]:
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError(
        "No se encontró PINECONE_API_KEY en los Secrets de Colab."
    )

print("API key de Pinecone cargada correctamente.")

API key de Pinecone cargada correctamente.


## Conexión con Pinecone

En esta celda me conecto con el índice de Pinecone que creé específicamente para este proyecto.

Este índice contiene los embeddings de los chunks del corpus de ARCA.

In [7]:
PINECONE_INDEX_NAME = "arca-monotributo"

pc = Pinecone(
    api_key=PINECONE_API_KEY
)

index = pc.Index(
    PINECONE_INDEX_NAME
)

print("Conectado correctamente con Pinecone.")
print(f"Índice utilizado: {PINECONE_INDEX_NAME}")

Conectado correctamente con Pinecone.
Índice utilizado: arca-monotributo


## Verificación del índice

En esta celda consulto las estadísticas del índice de Pinecone.

Quiero comprobar que los vectores que cargué en el notebook anterior están efectivamente almacenados en el índice.

In [8]:
estadisticas = index.describe_index_stats()

print(estadisticas)

DescribeIndexStatsResponse(dimension=384, total_vector_count=43, metric='cosine', namespaces=1)


## Carga del modelo de embeddings

En esta celda cargo el mismo modelo que utilicé para generar los embeddings de los documentos.

Es importante utilizar el mismo modelo para los documentos y para las preguntas, porque necesito que ambos tipos de texto estén representados dentro del mismo espacio vectorial.

In [9]:
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

modelo_embeddings = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Modelo de embeddings cargado correctamente.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo de embeddings cargado correctamente.


## Parámetros del retrieval

En esta celda defino la cantidad de resultados que quiero recuperar de Pinecone.

Por ahora voy a recuperar los 5 chunks más similares.

Más adelante, cuando construya el RAG completo, voy a analizar si conviene utilizar un número fijo de documentos o aplicar además un umbral de similitud.

In [10]:
TOP_K = 5

print(f"Cantidad de documentos a recuperar: {TOP_K}")

Cantidad de documentos a recuperar: 5


## Función de retrieval

En esta celda creo una función que recibe una pregunta y realiza el retrieval.

Primero genero el embedding de la pregunta utilizando el mismo modelo utilizado para los documentos.

Después envío ese vector a Pinecone y solicito los `TOP_K` vectores más similares.

También solicito los metadatos porque allí tengo almacenado el texto original de cada chunk.

In [11]:
def recuperar_documentos(pregunta):

    embedding_pregunta = modelo_embeddings.encode(
        pregunta
    ).tolist()

    resultado = index.query(
        vector=embedding_pregunta,
        top_k=TOP_K,
        include_metadata=True
    )

    return resultado.matches

## Primera prueba de retrieval

En esta celda pruebo el retrieval con una pregunta relacionada con uno de los trámites del corpus.

Todavía no genero una respuesta.

Solamente quiero observar qué fragmentos considera relevantes Pinecone.

In [12]:
pregunta = "¿Cómo puedo obtener la clave fiscal?"

matches = recuperar_documentos(
    pregunta
)

print(
    f"Documentos recuperados: {len(matches)}"
)

Documentos recuperados: 5


## Visualización de los documentos recuperados

En esta celda muestro los documentos recuperados por Pinecone.

Para cada resultado muestro el score de similitud, el documento de origen, el número de chunk y el texto.

Esto me permite evaluar manualmente si el retrieval está encontrando información relevante para la pregunta.

In [13]:
print("=" * 80)
print("DOCUMENTOS RECUPERADOS")
print("=" * 80)

for i, match in enumerate(matches, start=1):

    metadata = match.metadata

    print(f"\nDocumento {i}")
    print(f"Score: {match.score:.4f}")

    print(
        f"Archivo: {metadata.get('archivo', 'desconocido')}"
    )

    print(
        f"Documento ID: {metadata.get('documento_id', 'desconocido')}"
    )

    print(
        f"Chunk: {metadata.get('chunk', 'desconocido')}"
    )

    print("\nTexto:")
    print(
        metadata.get("texto", "")
    )

    print("\n" + "-" * 80)

DOCUMENTOS RECUPERADOS

Documento 1
Score: 0.6147
Archivo: corpus_arca_monotributo.json
Documento ID: inicio
Chunk: 1

Texto:
egorización
FINALIZACIÓN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
Exclusión
Renuncia
Pasaje al régimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder después darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener la
clave fiscal
y la
CUIT
.
Luego, hay que
ingresar con clave fiscal
y habilitar el
domicilio fiscal electrónico
, indispensable para recibir las comunicaciones oficiales de esta Agencia de Recaudación y
evitar estafas
.
Una vez realizadas estas gestiones, se debe acceder al
servicio con clave fiscal "Registro Único Tributario – RUT"
para ingresar la información de los
domicilios
, declarar las
actividades a desarrollar
y comenzar el proceso de alta en el monotributo.
Todos los trámites necesarios

-------------------------------------------------------------------------

## Segunda prueba de retrieval

En esta celda pruebo una pregunta relacionada con la recategorización del Monotributo.

Quiero comprobar que el retrieval cambia según la pregunta y que los chunks recuperados están relacionados semánticamente con el tema consultado.

In [14]:
pregunta = "¿Cómo hago la recategorización del monotributo?"

matches = recuperar_documentos(
    pregunta
)

print("=" * 80)
print(f"PREGUNTA: {pregunta}")
print("=" * 80)

for i, match in enumerate(matches, start=1):

    metadata = match.metadata

    print(f"\nDocumento {i}")
    print(f"Score: {match.score:.4f}")

    print(
        f"Archivo: {metadata.get('archivo', 'desconocido')}"
    )

    print(
        f"Chunk: {metadata.get('chunk', 'desconocido')}"
    )

    print("\nTexto:")
    print(
        metadata.get("texto", "")
    )

    print("\n" + "-" * 80)

PREGUNTA: ¿Cómo hago la recategorización del monotributo?

Documento 1
Score: 0.7184
Archivo: corpus_arca_monotributo.json
Chunk: 23

Texto:
la misma categoría
.
Recategorización simplificada
Mediante este procedimiento, al iniciar el trámite en el portal Monotributo, el sistema muestra
automáticamente la facturación anual
, facilitando el proceso.
Los pasos detallados para recategorizarse pueden consultarse en la
guía "Recategorización de Monotributo"
.
Recategorización de oficio
La recategorización de oficio es el proceso que aplica ARCA cuando un contribuyente del Monotributo no se recategoriza o lo hace de manera incorrecta.
Este procedimiento se inicia cuando ARCA detecta que las compras, gastos o acreditaciones bancarias del contribuyente superan el límite máximo de ingresos brutos anuales permitido para la categoría en la que se encuentra inscripto.
Notificación
Las personas recategorizadas de oficio reciben una notific

----------------------------------------------------------

## Tercera prueba: pregunta fuera del corpus

En esta celda realizo una pregunta que no está relacionada con ARCA ni con el Monotributo.

Quiero observar qué sucede cuando Pinecone recibe una pregunta para la cual mi corpus no contiene información relevante.

Esta prueba será importante más adelante porque el RAG deberá poder reconocer cuándo no tiene información suficiente para responder.

In [15]:
pregunta = "¿Cuál es la capital de Francia?"

matches = recuperar_documentos(
    pregunta
)

print("=" * 80)
print(f"PREGUNTA: {pregunta}")
print("=" * 80)

for i, match in enumerate(matches, start=1):

    metadata = match.metadata

    print(f"\nDocumento {i}")
    print(f"Score: {match.score:.4f}")

    print(
        f"Archivo: {metadata.get('archivo', 'desconocido')}"
    )

    print(
        f"Chunk: {metadata.get('chunk', 'desconocido')}"
    )

    print("\nTexto:")
    print(
        metadata.get("texto", "")
    )

    print("\n" + "-" * 80)

PREGUNTA: ¿Cuál es la capital de Francia?

Documento 1
Score: 0.4011
Archivo: corpus_arca_monotributo.json
Chunk: 8

Texto:
de comprobantes

--------------------------------------------------------------------------------

Documento 2
Score: 0.3984
Archivo: corpus_arca_monotributo.json
Chunk: 6

Texto:
a presentarte en una
dependencia de ARCA
. El día del turno tendrás que llevar:
el
formulario 206 – Multinota
, donde manifiestes tu voluntad de solicitar la clave fiscal y el documento que acredite tu identidad, según corresponda:
Argentinos nativos o naturalizados:
original y fotocopia del documento nacional de identidad (DNI) vigente.
Extranjeros:
original y fotocopia del documento de identidad del país de origen, pasaporte o cédula del MERCOSUR (de tratarse de un país limítrofe).
Extranjeros con residencia en el país (incluida la temporaria o transitoria) que no posean documento nacional de identidad:
original y fotocopia de la cédula de identidad, certificado o comprobante que acred

## Construcción del contexto

En esta celda creo una función que toma los documentos recuperados y construye un único texto.

Este texto será el contexto que posteriormente voy a entregar al modelo de lenguaje para generar la respuesta del RAG.

In [16]:
def construir_contexto(matches):

    bloques = []

    for i, match in enumerate(matches, start=1):

        texto = match.metadata.get(
            "texto",
            ""
        ).strip()

        if texto:

            bloques.append(
                f"[Documento {i}]\n{texto}"
            )

    return "\n\n---\n\n".join(
        bloques
    )

## Construcción del contexto para una pregunta

En esta celda realizo nuevamente una consulta sobre un trámite de ARCA.

Después de recuperar los documentos, construyo el contexto que posteriormente utilizaré como entrada del modelo generativo.

Por ahora solamente muestro el contexto para verificar qué información llegará al LLM.

In [17]:
pregunta = "¿Cómo puedo obtener la clave fiscal?"

matches = recuperar_documentos(
    pregunta
)

contexto = construir_contexto(
    matches
)

print("=" * 80)
print("CONTEXTO RECUPERADO")
print("=" * 80)

print(contexto)

CONTEXTO RECUPERADO
[Documento 1]
egorización
FINALIZACIÓN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
Exclusión
Renuncia
Pasaje al régimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder después darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener la
clave fiscal
y la
CUIT
.
Luego, hay que
ingresar con clave fiscal
y habilitar el
domicilio fiscal electrónico
, indispensable para recibir las comunicaciones oficiales de esta Agencia de Recaudación y
evitar estafas
.
Una vez realizadas estas gestiones, se debe acceder al
servicio con clave fiscal "Registro Único Tributario – RUT"
para ingresar la información de los
domicilios
, declarar las
actividades a desarrollar
y comenzar el proceso de alta en el monotributo.
Todos los trámites necesarios

---

[Documento 2]
Pagos
Recategorización
FINALIZACIÓN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
Exclusión
Renuncia
Pasaje al régimen general
Ayuda
Obten

## Inspección de los scores

En esta celda muestro solamente los scores de similitud obtenidos para la pregunta.

Quiero observar la diferencia entre los resultados más relevantes y los menos relevantes.

Estos valores me servirán posteriormente para decidir si necesito aplicar un umbral de similitud antes de enviar los documentos al modelo generativo.

In [18]:
print("=" * 80)
print("SCORES DE SIMILITUD")
print("=" * 80)

for i, match in enumerate(matches, start=1):

    print(
        f"Documento {i}: "
        f"{match.score:.4f}"
    )

SCORES DE SIMILITUD
Documento 1: 0.6147
Documento 2: 0.5985
Documento 3: 0.5872
Documento 4: 0.5835
Documento 5: 0.5828


## Conclusión

En este notebook implementé la etapa de retrieval de mi sistema RAG.

El proceso que realicé fue:

1. Generé el embedding de una pregunta.
2. Consulté Pinecone utilizando ese embedding.
3. Recuperé los chunks más similares.
4. Inspeccioné sus scores y textos.
5. Construí el contexto que posteriormente recibirá el modelo generativo.

Con esto ya tengo funcionando la etapa de recuperación de información.

En el próximo notebook voy a incorporar el modelo de lenguaje para completar el flujo RAG:

Pregunta → Embedding → Pinecone → Contexto → LLM → Respuesta.

El objetivo es que el modelo genere respuestas utilizando únicamente la información recuperada desde el corpus oficial de ARCA.